# COS30049 A2 - The randomized test set: both Explorer Assistants, both rounds (T4 x2)

Scores Qwen 3B (round 2 and round 1) and the 152M model (round 1) on the randomized test set
(`data/processed/random_test/`, `explorer/random_test.py`): 500 real prompts drawn from DS7 v2's held-out addresses,
500 synthetic transactions no writer has seen, 500 hard ones (any of the nine methods, reworded instructions,
shuffled facts), 883 real ones balanced by case (DS10 and DS2's test split), and the sets of 27 Oct: one balanced
set per crawled source (DS1, DS2, DS10), close calls, extreme values and maximal shift. The template's own rows were scored
when the set was built; the 152M model's round 2 is scored by `cos30049a2-text150m-stages`. The ten sets are shared
between the GPUs, each set on one GPU only, so each summary table has one writer process.

**Settings:** Accelerator **GPU T4 x2**, **Internet on**. **Inputs:** this account's notebooks
`cos30049a2-qwen3b-explain-v2` (round 2's adapter) and `cos30049a2-qwen3b-explain` (round 1's), and dataset
`cos30049a2-text150m-base` (the 152M round-1 writer). Then **Save Version -> Save & Run All (Commit)**; about 3 hours.

In [ ]:
REPO = "https://github.com/Defaul119/COS30049A2.git"
COMMIT = "028ec4c2209f784ec2abba49cc1a5f73a59e7cd6"   # the code and the test set this run uses; recorded in kaggle_run.json
PINS = ["transformers==4.52.4", "peft==0.15.2", "bitsandbytes==0.48.1", "accelerate==1.7.0", "tokenizers==0.21.1"]
EVAL_BATCH = 32       # prompts Qwen answers together

In [ ]:
import contextlib, hashlib, json, os, shutil, subprocess, sys, threading, time
from importlib.metadata import version
from pathlib import Path

T0 = time.time()                           # the session's clock: Kaggle stops a session 12 h after it starts
SRC = Path("/tmp/COS30049A2")             # the code and the data it works on; not part of the output
WORK = Path("/kaggle/working")             # the notebook's output


def sh(*cmd, log=None, cwd=None, env=None):
    """Run a command, print its output as it comes, keep a copy in WORK/<log>; raise if it fails."""
    with contextlib.ExitStack() as stack:
        copy = stack.enter_context(open(WORK / log, "a", encoding="utf-8")) if log else None
        p = stack.enter_context(subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                                 text=True, bufsize=1, env=env))
        for line in p.stdout:
            print(line, end="", flush=True)
            if copy:
                copy.write(line)
                copy.flush()
    if p.returncode:
        raise subprocess.CalledProcessError(p.returncode, cmd)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


# Kaggle grants internet (and GPUs) only to phone-verified accounts, and otherwise switches them off silently.
import socket
try:
    socket.create_connection(("github.com", 443), timeout=10).close()
except OSError:
    raise SystemExit("No internet in this session: verify a phone number (kaggle.com -> Settings -> Phone "
                     "verification), switch Internet on in the notebook's Settings panel, and run again.")

if shutil.which("nvidia-smi") is None:
    raise SystemExit("No GPU in this session: Settings panel -> Accelerator -> GPU T4 x2 (needs a verified phone).")
GPUS = len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines())

if not SRC.exists():
    sh("git", "clone", "--quiet", REPO, str(SRC))
sh("git", "checkout", "--quiet", COMMIT, cwd=SRC)
sh(sys.executable, "-m", "pip", "install", "--quiet", *PINS)
os.chdir(SRC)

def find_input(rel):
    """The file under /kaggle/input whose path ends with rel (inputs mount under their own folder names)."""
    hits = [p for p in Path("/kaggle/input").rglob(Path(rel).name) if p.as_posix().endswith(rel)]
    if not hits:
        raise SystemExit(f"{rel} is not in any input: add the inputs named at the top")
    return hits[0]


def check_base():
    """The round-1 files (dataset cos30049a2-text150m-base), each checked against the dataset's SHA256SUMS.txt."""
    sums = find_input("SHA256SUMS.txt")
    for line in sums.read_text().splitlines():
        digest, name = line.split(maxsplit=1)
        assert sha256(sums.parent / name.lstrip("*")) == digest, f"{name} differs from SHA256SUMS.txt"
    return sums.parent

In [ ]:
base = check_base()
V2_ADAPTER = find_input("runs/qwen3b-explain-v2/adapter_best/adapter_config.json").parent
V1_ADAPTER = find_input("runs/qwen3b-explain/adapter_best/adapter_config.json").parent
RANDOM = SRC / "data" / "processed" / "random_test"
manifest = json.loads((RANDOM / "manifest.json").read_text(encoding="utf-8"))
for f in manifest["files"]:
    assert sha256(SRC / f["path"]) == f["sha256"], f"{f['path']} differs from its manifest"
WRITERS = [("qwen3b_v2", ["--config", str(SRC / "configs" / "qwen3b_explain_v2.json"), "--adapter", str(V2_ADAPTER),
                          "--batch-size", str(EVAL_BATCH)]),
           ("qwen3b", ["--config", str(SRC / "configs" / "qwen3b_explain.json"), "--adapter", str(V1_ADAPTER),
                       "--batch-size", str(EVAL_BATCH)]),
           ("text150m", ["--checkpoint", str(base / "text150m_explain_v1.pt")])]
QUEUES = [[(w, a, part) for part in ("real", "hard", "crawl_ds1", "crawl_ds10", "extreme") for w, a in WRITERS],
          [(w, a, part) for part in ("synthetic", "balanced", "crawl_ds2", "close_calls", "max_shift") for w, a in WRITERS]]
if GPUS == 1:
    QUEUES = [QUEUES[0] + QUEUES[1]]


def run_queue(gpu, queue):
    for model, args, part in queue:
        # --n above every set's size: each prompt is answered (the default draws 500; balanced has 883)
        sh(sys.executable, "-m", "lm.eval_explain", "--model", model, *args, "--test", str(RANDOM / f"{part}.jsonl"),
           "--tag", f"random_{part}", "--n", "100000", log=f"eval-{model}-random_{part}.log",
           env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)))


threads = [threading.Thread(target=run_queue, args=(gpu, q)) for gpu, q in enumerate(QUEUES)]
for t in threads:
    t.start()
for t in threads:
    t.join()
tables = WORK / "reports" / "tables"
tables.mkdir(parents=True, exist_ok=True)
for f in (SRC / "reports" / "tables").glob("assistant_eval*random*.csv"):
    shutil.copy(f, tables)
(WORK / "kaggle_run.json").write_text(json.dumps({
    "commit": COMMIT, "gpus": GPUS, "python": sys.version.split()[0], "session_hours": round((time.time() - T0) / 3600, 2),
    "packages": {p: version(p) for p in ("torch", "transformers", "peft", "bitsandbytes", "tokenizers")},
}, indent=2), encoding="utf-8")
for p in sorted(tables.iterdir()):
    print(p.name)